## Setup & Imports
PyTorch, `nn`, `transforms`, `DataLoader` import kiye. `SqueezeNet` (torchvision ka pre-built model) bhi import kiya hai jo baad mein use hoga. `helper_utils` module bhi import kiya jisme dataset load karne ka helper function hai.

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import SqueezeNet
import helper_utils

## Dataset Load Karna
`helper_utils.get_dataset()` se dataset load kiya (ye Fashion MNIST hai, jaisa aage ke code se pata chalta hai). `transforms.ToTensor()` se images ko PyTorch tensor mein convert karne ka transform set kiya.

In [4]:
dataset = helper_utils.get_dataset()

transform = transforms.ToTensor()
dataset.transform = transform

Dataset already exists.


## DataLoader Banana
Dataset ko `DataLoader` mein daala, batch size 64 ke saath. `shuffle=False` hai — matlab data ka order fix rahega, random nahi hoga.

In [5]:
batch_size = 64
dataloader = DataLoader(dataset, batch_size, shuffle=False)

## Ek Batch Check Karna
`next(iter(dataloader))` se dataloader ka pehla batch nikaala aur uska shape print kiya — taaki pata chale ki images kis shape (batch_size, channels, height, width) mein aa rahi hain.

In [6]:
image_batch, label_batch = next(iter(dataloader))
print("Batch Shape : ", image_batch.shape)

Batch Shape :  torch.Size([64, 1, 28, 28])


## Model Banaya (SimpleCNN) — Pehla Version
Ek basic CNN define kiya: `Conv2d` (1 input channel, 32 output channels) → `ReLU` → `MaxPool2d`, phir 2 Fully Connected layers (`fc1`, `fc2`) jo 10 classes (Fashion MNIST) predict karte hain.

**Important**: `forward()` mein conv/pool ke baad seedha `fc1` mein data bhej diya hai — **flatten step missing hai**, jo aage error dega.

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional Block
        self.conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Block
        # For Fashion MNIST: input images are 28x28,
        # after conv+pool: 32x14x14
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  # 10 classes for Fashion MNIST

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)
        return x

## Model Test Kiya — Error Aayega
Model ko ek image batch pe run karne ki koshish ki. Kyunki `forward()` mein **flatten step missing hai** (conv/pool ka 4D output seedha Linear layer ko de diya, jo 2D expect karta hai), ye **error dega** — ye jaan-bujh kar dikhaya gaya bug hai taaki samajh sake flatten kyu zaroori hai.

In [9]:
simple_cnn = SimpleCNN()

try:
    output = simple_cnn(image_batch)  
except Exception as e:
    print(f"\033[91mError during forward pass: {e}\033[0m")

Error during forward pass: mat1 and mat2 shapes cannot be multiplied (28672x14 and 6272x128)


## Debug Version Banaya
`SimpleCNN` ko inherit karke `SimpleCNNDebug` banaya jisme `forward()` mein har step pe `print()` statements daale hain — taaki dekh sake har layer ke baad tensor ka shape aur weights/biases ka shape kya hai. Isse pata chalega exactly kaha error aa raha hai.

In [10]:
class SimpleCNNDebug(SimpleCNN):
    def __init__(self):
        super().__init__()
        # The super().__init__() call above properly initializes all layers from SimpleCNN
        # No need to redefine the layers here

    def forward(self, x):
        print("Input shape:", x.shape)
        print(
            " (Layer components) Conv layer parameters (weights, biases):",
            self.conv.weight.shape,
            self.conv.bias.shape,
        )
        x_conv = self.relu(self.conv(x))

        print("===")

        print("(Activation) After convolution and ReLU:", x_conv.shape)
        x_pool = self.pool(x_conv)
        print("(Activation) After pooling:", x_pool.shape)

        print(
            "(Layer components) Linear layer fc1 parameters (weights, biases):",
            self.fc1.weight.shape,
            self.fc1.bias.shape,
        )

        x_fc1 = self.relu_fc(self.fc1(x_pool))

        print("===")

        print("(Activation) After fc1 and ReLU:", x_fc1.shape)

        print(
            "(Layer components) Linear layer fc2 parameters (weights, biases):",
            self.fc2.weight.shape,
            self.fc2.bias.shape,
        )
        x = self.fc2(x_fc1)

        print("===")

        print("(Activation) After fc2 (output):", x.shape)
        return x

## Debug Model Run Kiya
Debug version ko run kiya taaki print statements se pata chale ki conv aur pooling ke baad tensor 4D hai (batch, channels, height, width), jabki `fc1` (Linear layer) 2D input expect karta hai — yahi mismatch error ka reason hai.

In [13]:
simple_cnn_debug = SimpleCNNDebug()

try:
    output_debug = simple_cnn_debug(image_batch)  
except Exception as e:
    print(f"\033[91mError during forward pass in debug model: {e}\033[0m")

Input shape: torch.Size([64, 1, 28, 28])
 (Layer components) Conv layer parameters (weights, biases): torch.Size([32, 1, 3, 3]) torch.Size([32])
===
(Activation) After convolution and ReLU: torch.Size([64, 32, 28, 28])
(Activation) After pooling: torch.Size([64, 32, 14, 14])
(Layer components) Linear layer fc1 parameters (weights, biases): torch.Size([128, 6272]) torch.Size([128])
Error during forward pass in debug model: mat1 and mat2 shapes cannot be multiplied (28672x14 and 6272x128)


## Fixed Version Banaya
`SimpleCNNFixed` mein bug fix kiya — conv+pool ke baad `torch.flatten(x_pool, start_dim=1)` add kiya, jo 4D tensor ko 2D mein convert karta hai (batch dimension chhod ke baaki sab ek row mein flatten) — taaki Linear layer ko sahi shape mile.

In [15]:
class SimpleCNNFixed(SimpleCNN):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        print("Input shape:", x.shape)
        print(
            " (Neuron components) Conv layer parameters (weights, biases):",
            self.conv.weight.shape,
            self.conv.bias.shape,
        )
        x_conv = self.relu(self.conv(x))

        print("===")

        print("(Activation) After convolution and ReLU:", x_conv.shape)
        x_pool = self.pool(x_conv)
        print("(Activation) After pooling:", x_pool.shape)

        x_flattened = torch.flatten(
            x_pool, start_dim=1
        )  # Flatten all dimensions except batch
        print("(Activation) After flattening:", x_flattened.shape)

        print(
            "(Neuron components) Linear layer fc1 parameters (weights, biases):",
            self.fc1.weight.shape,
            self.fc1.bias.shape,
        )

        x_fc1 = self.relu_fc(self.fc1(x_flattened))

        print("===")

        print("(Activation) After fc1 and ReLU:", x_fc1.shape)

        print(
            "(Neuron components) Linear layer fc2 parameters (weights, biases):",
            self.fc2.weight.shape,
            self.fc2.bias.shape,
        )
        x = self.fc2(x_fc1)

        print("===")

        print("(Activation) After fc2 (output):", x.shape)
        return x

## Fixed Model Test Kiya
Fixed model ko run kiya — ab bina error ke chalega, kyunki flatten step add ho chuka hai.

In [16]:
simple_cnn_fixed = SimpleCNNFixed()
output = simple_cnn_fixed(image_batch)

Input shape: torch.Size([64, 1, 28, 28])
 (Neuron components) Conv layer parameters (weights, biases): torch.Size([32, 1, 3, 3]) torch.Size([32])
===
(Activation) After convolution and ReLU: torch.Size([64, 32, 28, 28])
(Activation) After pooling: torch.Size([64, 32, 14, 14])
(Activation) After flattening: torch.Size([64, 6272])
(Neuron components) Linear layer fc1 parameters (weights, biases): torch.Size([128, 6272]) torch.Size([128])
===
(Activation) After fc1 and ReLU: torch.Size([64, 128])
(Neuron components) Linear layer fc2 parameters (weights, biases): torch.Size([10, 128]) torch.Size([10])
===
(Activation) After fc2 (output): torch.Size([64, 10])


## Model Sequential Style Mein Likha
Same architecture ko `nn.Sequential` use karke zyada compact tareeke se likha — `conv_block` (Conv+ReLU+Pool) aur `fc_block` (Linear+ReLU+Linear) do groups mein organize kiya. `forward()` mein bas in blocks ko call karna hai aur beech mein `flatten` lagana hai.

In [17]:
class SimpleCNN2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional Block
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Fully Connected Block
        # For Fashion MNIST: input images are 28x28,
        # after conv+pool: 32x14x14
        flattened_size = 32 * 14 * 14
        self.fc_block = nn.Sequential(
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Linear(128, 10),  # 10 classes for Fashion MNIST
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = torch.flatten(x, start_dim=1)  # Flatten all dimensions except batch
        x = self.fc_block(x)
        return x

## Sequential Model Test Kiya
Sequential version ko run kiya aur output shape print kiya — confirm karne ke liye ki ye bhi sahi kaam kar raha hai.

In [18]:
simple_cnn_seq = SimpleCNN2Seq()
output = simple_cnn_seq(image_batch)
print("Output shape from sequential model:")
print(output.shape)

Output shape from sequential model:
torch.Size([64, 10])


## Activation Statistics Dekhne Wala Debug Model
`SimpleCNN2SeqDebug` banaya jisme `get_statistics()` function hai jo kisi bhi layer ke output (activation) ka **mean, std, min, max** calculate karta hai. Ye samajhne mein help karta hai ki data model ke andar kaise distribute ho raha hai har stage pe (jaise agar values bahut bade/chhote ho rahe hain to training mein dikkat aa sakti hai).

In [19]:
class SimpleCNN2SeqDebug(SimpleCNN2Seq):
    def __init__(self):
        super().__init__()
        # The super().__init__() call above properly initializes all layers from SimpleCNN2Seq
        # No need to redefine the layers here

    def get_statistics(self, activation):
        mean = activation.mean().item()
        std = activation.std().item()
        min_val = activation.min().item()
        max_val = activation.max().item()

        print(f" Mean: {mean}")
        print(f" Std: {std}")
        print(f" Min: {min_val}")
        print(f" Max: {max_val}")
        return mean, std, min_val, max_val

    def forward(self, x):
        features = self.conv_block(x)
        x = torch.flatten(features, start_dim=1)  # Flatten all dimensions except batch

        print("After conv_block, the activation statistics are:")
        self.get_statistics(features)

        x = self.fc_block(x)
        print("After fc_block, the activation statistics are:")
        self.get_statistics(x)
        return x

## Multiple Batches pe Statistics Check Kiya
Dataloader se pehle 5 batches liye, aur har batch ke liye conv_block aur fc_block ke baad activation statistics print kiye — taaki dekh sake values training ke dauran kaisi rehti hain.

In [20]:
simple_cnn_seq_debug = SimpleCNN2SeqDebug()

for idx, (img_batch, _) in enumerate(dataloader):
    if idx < 5:
        print(f"=== Batch {idx} ===")
        output_debug = simple_cnn_seq_debug(img_batch)

=== Batch 0 ===
After conv_block, the activation statistics are:
 Mean: 0.21706977486610413
 Std: 0.2609214186668396
 Min: 0.0
 Max: 1.4828746318817139
After fc_block, the activation statistics are:
 Mean: 0.013646373525261879
 Std: 0.06855521351099014
 Min: -0.20830851793289185
 Max: 0.1819915771484375
=== Batch 1 ===
After conv_block, the activation statistics are:
 Mean: 0.222072035074234
 Std: 0.26984140276908875
 Min: 0.0
 Max: 1.439536452293396
After fc_block, the activation statistics are:
 Mean: 0.012469884008169174
 Std: 0.07218676060438156
 Min: -0.2016526758670807
 Max: 0.2062610387802124
=== Batch 2 ===
After conv_block, the activation statistics are:
 Mean: 0.22139452397823334
 Std: 0.2690463960170746
 Min: 0.0
 Max: 1.4815665483474731
After fc_block, the activation statistics are:
 Mean: 0.012525856494903564
 Std: 0.0705641359090805
 Min: -0.20493000745773315
 Max: 0.18806907534599304
=== Batch 3 ===
After conv_block, the activation statistics are:
 Mean: 0.21832010149955

## Pre-built Model: SqueezeNet
Torchvision ka pehle se bana hua **SqueezeNet** model load kiya (bina pretrained weights ke, sirf architecture) aur uska pura structure print kiya — taaki dekh sake ek real-world/production-grade CNN kaisi dikhti hai (khud ke SimpleCNN se kaafi zyada complex).

In [21]:
complex_model = SqueezeNet()
print(complex_model)

SqueezeNet(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(7, 7), stride=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (3): Fire(
      (squeeze): Conv2d(96, 16, kernel_size=(1, 1), stride=(1, 1))
      (squeeze_activation): ReLU(inplace=True)
      (expand1x1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
      (expand1x1_activation): ReLU(inplace=True)
      (expand3x3): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (expand3x3_activation): ReLU(inplace=True)
    )
    (4): Fire(
      (squeeze): Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1))
      (squeeze_activation): ReLU(inplace=True)
      (expand1x1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
      (expand1x1_activation): ReLU(inplace=True)
      (expand3x3): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (expand3x3_activation): ReLU(inplace=True)
    )
    (5): Fire(
   

## SqueezeNet ke Blocks Explore Kiya
SqueezeNet ke har top-level block (`named_children()`) ka naam print kiya, aur har block ke andar kitni layers/sub-blocks hain wo bhi dikhaya — taaki samajh sake bade models kaise nested blocks mein organize hote hain.

In [22]:
for name, block in complex_model.named_children():
    print(f"Block {name} has a total of {len(list(block.children()))} layers:")
    
    for idx, layer in enumerate(block.children()):
        if len(list(layer.children())) == 0:
            print(f"\t {idx} - Layer {layer}")
        else:
            layer_name = layer._get_name()  # More user-friendly name
            print(f"\t {idx} - Sub-block {layer_name} with {len(list(layer.children()))} layers")   

Block features has a total of 13 layers:
	 0 - Layer Conv2d(3, 96, kernel_size=(7, 7), stride=(2, 2))
	 1 - Layer ReLU(inplace=True)
	 2 - Layer MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
	 3 - Sub-block Fire with 6 layers
	 4 - Sub-block Fire with 6 layers
	 5 - Sub-block Fire with 6 layers
	 6 - Layer MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
	 7 - Sub-block Fire with 6 layers
	 8 - Sub-block Fire with 6 layers
	 9 - Sub-block Fire with 6 layers
	 10 - Sub-block Fire with 6 layers
	 11 - Layer MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
	 12 - Sub-block Fire with 6 layers
Block classifier has a total of 4 layers:
	 0 - Layer Dropout(p=0.5, inplace=False)
	 1 - Layer Conv2d(512, 1000, kernel_size=(1, 1), stride=(1, 1))
	 2 - Layer ReLU(inplace=True)
	 3 - Layer AdaptiveAvgPool2d(output_size=(1, 1))


## Ek Specific Sub-block (Fire Module) Deep Dive
SqueezeNet ke `features[4]` (jo ek 'Fire module' hai — SqueezeNet ki khaas architecture unit) ke andar ke saare modules (layers) individually print kiye, taaki uski internal structure detail mein dikhe.

In [23]:
first_fire_module = complex_model.features[4]

for idx, module in enumerate(first_fire_module.modules()) :
    if idx > 0 :
        print(module)

Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1))
ReLU(inplace=True)
Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
ReLU(inplace=True)
Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
ReLU(inplace=True)
